# AI ANALYST LAB

![](../_img/ai_analyst_lab.png)

### A Hands-on Course on AI for Data Analysts
## Session 03: A/B testing in growth analytics

Feedback should be sent to [goran.milovanovic@datakolektiv.com](mailto:goran.milovanovic@datakolektiv.com).

This notebook accompanies the **AI Analyst LAB** course. Welcome to Session 03 — your first week running real A/B tests.

### Lecturer

[Goran S. Milovanović, PhD, DataKolektiv, Chief Scientist & Owner](https://www.linkedin.com/in/gmilovanovic/)

***
### What we will do today

Two weeks ago (Session 01) we summarized data with descriptive statistics and learned how to put a number on the uncertainty in a mean (the CLT and the standard error). Last week (Session 02) we developed the language of probability — sample space, events, conditional probability, expected value — and used the bootstrap to put confidence intervals on any statistic we cared to compute. This week we sharpen those tools into a single workflow: the **hypothesis testing framework**.

The business setting this week is a growth team running A/B tests on headlines. Two headlines compete; one of them wins more clicks. Did it really win, or were we just unlucky with the variant that lost? *That* is the question hypothesis testing was built to answer.

We will move from intuition to formal mechanics, then bring it all back to a real dataset (the **Upworthy Research Archive** — 22,000 actual headline tests run between 2013 and 2015), then finish with a structured **A/B test analysis plan** and a stakeholder memo. The sections, in order:

| Section | What happens |
|---|---|
| 3.1 | The business case — what you are being asked to deliver this week |
| 3.2 | Meet your Session 03 Tutor (Claude Project) |
| 3.3 | Setup — imports and loading the Upworthy data |
| 3.4 | First look — what is actually in the file? |
| 3.5 | The hypothesis testing framework — intuition first |
| 3.6 | The chi-square test for proportions (the right tool for A/B tests on clicks) |
| 3.7 | The two-sample t-test for comparing means (the right tool for continuous outcomes) |
| 3.8 | Confidence intervals — the parametric companion to bootstrap CIs |
| 3.9 | p-hacking and the multiple-testing problem |
| 3.10 | Statistical power — why "no significant difference" is not the same as "no difference" |
| 3.11 | Practical vs statistical significance — the most important distinction in the session |
| 3.12 | Our API calls — Anthropic tool use for the A/B test analysis plan |
| 3.13 | The A/B test analysis plan and stakeholder memo — fully worked |
| 3.14 | References — what to study to deepen this session |

A few rules for using this notebook, same as the previous two weeks:

- **Run the cells in order.** Each section builds on the one before.
- **Read the explanations, do not just run the cells.** The intuition is the point; the formulas are the receipts.
- **Every line of code has a comment above it** in beginner language.
- **Use your Session 03 Tutor** (the Claude Project at `_tutors/session03_tutor.xml`) when something feels confusing.
- **Compute first in Python, then use the model to interpret.** Same rule as Sessions 01 and 02 — the model never invents numbers.

***
## 3.1 The business case

You have just moved teams again. This week you are embedded with a **growth team at a digital media company**. The team's job is to maximize how many readers click on the stories the editorial team publishes — without sensationalizing the journalism — and the way they do this is by running **A/B tests on headlines**. For every story, the team writes two (or more) candidate headlines. Each variant is shown to a slice of visitors. Whichever variant earns more clicks per impression wins, and that one ships site-wide.

Your manager hands you a deck of last quarter's tests and the brief:

> *"Some of these tests have a 'winner' label on them, but I do not trust the label process. For this Friday I need: **(a)** an A/B test analysis plan — what we measure, what null and alternative we test, what counts as winning — and **(b)** a stakeholder memo for one specific test we are deciding whether to ship. Be honest about what we know and what we do not."*

This is the most common job in modern web analytics. *"Did B beat A?"* sounds simple. It is not. With small enough samples, very large differences can happen purely by chance. With large enough samples, vanishingly small differences can be "statistically significant" without being practically useful. With many concurrent tests, *some* of them will look like winners even when no real effect exists. The hypothesis testing framework is how analysts navigate all of this without overclaiming.

By the end of this session you will have:

1. **Loaded** the Upworthy Research Archive and applied the data-limitations discipline (excluding the well-known randomization window).
2. **Built** the full intuition for hypothesis testing — null, alternative, decision rule, p-value, the role of the sampling distribution.
3. **Run a real chi-square test** on one specific A/B test from the archive and interpreted the p-value honestly.
4. **Computed a t-test** for a different kind of outcome (comparing means rather than proportions).
5. **Computed confidence intervals** for both kinds of outcomes — and seen how they relate to last week's bootstrap CIs.
6. **Demonstrated multiple-testing risk** by running tests across many comparisons at once.
7. **Discussed statistical power** and **practical significance** — the two ideas that prevent the most common over-claiming mistakes.
8. **Used Anthropic tool use** with a Pydantic schema to generate a schema-enforced A/B test analysis plan, and a second Anthropic call to draft the stakeholder memo.
9. **Delivered both artifacts** — the plan and the memo — grounded entirely in numbers computed in this notebook.

Two threads weave through the whole session:

- **How do we frame the question rigorously?** — sections 3.5 through 3.8.
- **How do we resist over-claiming?** — sections 3.9 through 3.11.

The artifacts in section 3.13 bring both threads together. Let's get started.

***
## 3.2 Meet your Session 03 Tutor (Claude Project)

Before you continue, you should have set up your **Session 03 Tutor** — a Claude Project configured to teach you the hypothesis-testing ideas in this notebook in a gentle, beginner-friendly way.

If you have not done this yet, open **[`_tutors/TutorProjectCreation.md`](../_tutors/TutorProjectCreation.md)** and follow the steps using **[`_tutors/session03_tutor.xml`](../_tutors/session03_tutor.xml)** as the project's instructions. It takes about five minutes.

The tutor knows you have finished Sessions 01 and 02 and that you are working on the Upworthy A/B test brief this week. It is configured to make Session 01 and 02 callbacks where they help. It also knows about its siblings: ask it Python questions and it will redirect you to `python_stack_tutor`; ask it PowerShell questions and it will redirect you to `windows_powershell_tutor`.

Some example questions you might paste into the tutor while working through this notebook:

- *"My A/B test came back with p = 0.04 — does that mean B is really the winner? What else should I check?"*
- *"What is the difference between 'failing to reject the null' and 'accepting the null'? Aren't those the same thing?"*
- *"Why is the chi-square test right one for click-through rates and not a t-test?"*
- *"I ran 20 tests this quarter, three came back p < 0.05. Should I be excited or worried?"*

***
## 3.3 Setup — imports and loading the data

Same opening move as the previous two weeks: imports first, data second, sanity-check third.

> **Callback to §1.3 and §2.3.** Four libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`) plus one new arrival: **`scipy.stats`**, which gives us the two-sample t-test and the chi-square test for free. We will introduce each `scipy.stats` function as we first use it. If you need a refresher on what `pandas` does (or any of the four library aliases), scroll back to Session 01 §1.3 or ask your `python_stack_tutor`.

In [ ]:
# Import pandas under the alias pd; pandas gives us the DataFrame (tables of data).
import pandas as pd

# Import numpy under the alias np; we will use it for arrays and random sampling (multiple-testing demo).
import numpy as np

# Import matplotlib's plotting module under the alias plt.
import matplotlib.pyplot as plt

# Import seaborn under the alias sns; used in the p-value-histogram visualization.
import seaborn as sns

# Import the `stats` submodule of SciPy; this contains ready-made hypothesis tests like ttest_ind and chi2_contingency.
from scipy import stats

# Jupyter magic that tells the notebook to display plots inline (directly in the notebook).
%matplotlib inline

# Cosmetic: clean seaborn default style.
sns.set_theme(style="whitegrid")

# Print a confirmation so we know the imports succeeded.
print("Libraries imported successfully.")

### The Upworthy Research Archive

This week's dataset is the **Upworthy Research Archive** — a published archive of A/B tests on headlines run by the media site Upworthy between 2013 and 2015, released openly under **CC BY 4.0** by Cornell University. The full archive contains *32,487 experiments and over 150,000 individual variants*. For this notebook we will use the **confirmatory subset**, which the archive maintainers designed for proper hypothesis-testing work.

Each row of the confirmatory file is one **variant** (the archive calls it a *"package"*) in an A/B test. Multiple variants share the same `clickability_test_id` — those rows together form one test. Key columns:

| Column | What it is |
|---|---|
| **`clickability_test_id`** | Unique identifier per A/B test. The thing we filter on to isolate one test. |
| **`headline`** | The variant's headline text shown to visitors. |
| **`impressions`** | How many visitors saw this variant. |
| **`clicks`** | How many of those visitors clicked. |
| `test_week` | Year + week the test ran, in `YYYYWW` format (e.g., `201446` = week 46 of 2014). |
| `significance`, `first_place`, `winner` | Upworthy's *own* internal labels. We will compute our own and ignore these. |

### Known caveat — the randomization window

The archive maintainers reported a randomization issue affecting tests deployed between **25 June 2013 and 10 January 2014**, and they advise excluding those tests from confirmatory analysis. We will filter them out using `test_week`. This is the *same data-limitations discipline* we built in §2.12 last week, just in a new context.

### Loading the confirmatory file

In [ ]:
# Read the confirmatory subset of the Upworthy archive into a DataFrame.
# low_memory=False reads the file in one pass so column types are inferred cleanly.
df = pd.read_csv(
    "../_data/upworthy_research_archive/upworthy-archive-confirmatory-packages-03.12.2020.csv",
    low_memory=False
)

# Filter out the randomization-issue window (test_week 201326 through 201402 inclusive).
# The condition keeps rows where test_week is BEFORE week 26 of 2013 OR AFTER week 2 of 2014.
mask_safe = (df["test_week"] < 201326) | (df["test_week"] > 201402)
df = df[mask_safe].copy()

# Print the number of rows before and after, plus the number of distinct A/B tests still in the data.
print(f"Rows in the safe-window subset : {len(df):,}")
print(f"Number of distinct tests       : {df['clickability_test_id'].nunique():,}")

**Expected output:**

```
Rows in the safe-window subset : 85,559
Number of distinct tests       : 17,817
```

About **86,000 variants** across **17,817 distinct A/B tests** survive the randomization-issue filter. That is plenty of material to demonstrate hypothesis testing — and to demonstrate, later, what happens when you naively run *many* tests at once.

> **Mini-recap of §3.3.** Imports done, Upworthy confirmatory data loaded, randomization-issue window excluded as the maintainers recommend. Every subsequent computation is on this filtered DataFrame `df`.

***
## 3.4 First look — what is actually in the file?

> **Callback to §1.4 and §2.4.** Trust the data before trusting the analysis. Four checks: shape, head, dtypes, missing values.

### Check 1 — shape

In [ ]:
# Show how many rows and columns we have after filtering.
df.shape

You should see `(85559, 17)` — about 86,000 variants, 17 columns each.

### Check 2 — head

Let us look at three rows so we can see what a variant actually looks like.

In [ ]:
# Peek at the first 3 rows; show only the columns we will actually use to keep the output readable.
df[["clickability_test_id", "headline", "impressions", "clicks", "test_week"]].head(10)

You will see three rows, each one a different headline variant. Each row has its own `impressions` and `clicks` counts; rows with the same `clickability_test_id` belong to the *same* A/B test and so are directly comparable.

### Check 3 — dtypes

In [ ]:
# Show the type of each column.
df.dtypes

`impressions`, `clicks`, and `test_week` are `int64` (whole numbers). `clickability_test_id` and `headline` are stored as text (`object`). That is all we need to know for now.

### Check 4 — missing values

In [ ]:
# Count missing values per column and only show columns where missingness is nonzero.
missing = df.isna().sum()
missing[missing > 0]

A few columns (`excerpt`, `lede`, `share_text`, `square`) have substantial missingness — those are auxiliary article metadata that we will not use for hypothesis testing. The columns we *will* use (`clickability_test_id`, `headline`, `impressions`, `clicks`, `test_week`) are complete.

### How many variants per test?

A/B tests in Upworthy did not always have exactly two variants. Some tests had three, four, or more competing headlines. Let us see the distribution.

In [ ]:
# Group rows by clickability_test_id and count rows per group; that is the number of variants per test.
variants_per_test = df.groupby("clickability_test_id").size()

# Show the distribution: how many tests had 2 variants, how many had 3, etc.
variants_per_test.value_counts().sort_index()

You will see:

```
2      475
3      606
4    7,798
5    3,940
6    3,930
7      593
8      319
9       72
10      35
11      22
12       7
13       6
14       7
15       3
17       3
20       1
```

The shape is more interesting than you might have guessed: the **classic 2-variant A/B test is actually a minority**. Only about 475 of the ~17,800 tests (under 3%) have exactly two variants. The most common structure is **4 variants** (about 7,800 tests — roughly 44% of the dataset), with 3, 5, and 6 variants also well represented. So in production, Upworthy almost always tested an A/B/C/D bake-off rather than a head-to-head pair.

For the rest of this notebook we will work with a **2-variant test** as the worked example because the pedagogy is cleanest with two groups. Just be aware that, in practice on this dataset, the more common setup is A/B/C/D — which raises the **multiple-testing** question even more sharply than two-variant tests do (we will get to it in §3.9).

***
## 3.5 The hypothesis testing framework — intuition first

Before we touch a formula, let us build the mental model. Hypothesis testing is one big idea wrapped in a small ritual. The big idea is this:

> **You cannot prove that an effect exists. You can only ask: *"how surprising would my data be if there were no effect?"* — and if the answer is *"very surprising"*, that is what passes for evidence in this framework.**

Read that sentence twice. It contains the entire framework.

### Why this framing?

Imagine your manager says: *"prove to me that headline B is better than headline A"*. You cannot. Even if every single one of B's impressions had become a click and every single one of A's had not, a sceptic could still ask: *"how do you know that was not just luck?"*

So we flip the burden of proof. We start by **assuming there is no real difference**, and we ask: *"under that assumption, how often would we see a CTR gap as big as the one we actually saw?"* If the answer is *"essentially never"*, the assumption of no-difference becomes hard to defend, and we have a *reason* to believe the difference is real. If the answer is *"oh, that happens all the time even by chance"*, we have learned nothing — the data is consistent with no real difference.

That is hypothesis testing. The ritual gives names to the pieces.

### The pieces

- **The null hypothesis $H_0$.** The "no real effect" position. In our A/B test, $H_0$ is *"headline A and headline B have the same true click-through rate"*. This is the position we are trying to *reject*.
- **The alternative hypothesis $H_1$.** The position we wish to be in. *"Headline A and B have different click-through rates"* (two-sided) or *"B has a higher click-through rate than A"* (one-sided). We use the two-sided version unless we have a strong directional prior.
- **The test statistic.** A number we compute from the data — chosen so that *if $H_0$ is true*, we know roughly how often each value of the statistic would occur. The chi-square statistic, the t-statistic, etc., are all *different choices of this number for different kinds of data*.
- **The sampling distribution under $H_0$.** The probability distribution of the test statistic *assuming the null is true*. This is what tells us *"how often would I see a value this extreme by chance?"*.
- **The p-value.** Formally:
  $$p = P(\text{test statistic at least as extreme as observed} \mid H_0 \text{ is true})$$
  In English: *"if there really were no effect, what fraction of repeated experiments would produce data this surprising or more?"*
- **The significance level $\alpha$.** A pre-committed threshold (commonly $0.05$ or $0.01$) below which we are willing to call the result *"too surprising under $H_0$ to keep believing $H_0$"*. We **pick $\alpha$ before we look at the data**.
- **The decision rule.** Reject $H_0$ if $p < \alpha$. Otherwise, *fail to reject* $H_0$.

> **Read this carefully.** *"Fail to reject $H_0$"* is **not the same** as *"accept $H_0$"*. It means the data did not have enough evidence to overturn the no-effect assumption. The effect might still be real but small, or real but hidden by limited sample size. **Absence of evidence is not evidence of absence.** This is the single most-violated rule in business reporting.

### Major callback to §1.9 — where the sampling distribution comes from

In §1.9 of Session 01 we did something that looks abstract now but matters here. We pretended the dataset was the population, **drew 10,000 fresh samples**, computed each sample's mean, and looked at the *distribution* of those 10,000 sample means. That distribution had a name: the **sampling distribution of the mean**.

The Central Limit Theorem told us that distribution was bell-shaped. The *width* of that bell — the standard error — was what told us how trustworthy any single sample's mean was.

The "sampling distribution under $H_0$" that we just defined is **exactly the same idea**. We imagine repeating the A/B test many times *assuming the null is true*, compute the test statistic each time, and look at the distribution of those test statistics. The p-value is the *fraction of those imaginary repetitions* that produce a test statistic as extreme as the one we actually got. Some test statistics (like the chi-square statistic) have a known mathematical form for this sampling distribution; some (like the bootstrap-based statistic from §2.11) require simulation. Either way the idea is identical.

### Two facts about p-values that contradict common intuition

- A p-value is **not** the probability that the null is true. It is a probability about the data *given* the null. To turn it into the probability the null is true, you would need Bayes' theorem (§2.6.7), a prior over hypotheses, and far more care than most reporting affords.
- A *small* p-value does **not** mean the effect is large. It means the data is unlikely under the null. With a huge sample, even a vanishingly small effect can produce a tiny p-value — which is what §3.11 will address.

> **Mini-recap of §3.5.** A hypothesis test asks *"how unlikely is the observed data under the no-effect assumption?"*. The answer is the p-value. The decision rule rejects the no-effect assumption if the p-value falls below a pre-committed threshold $\alpha$. *"Fail to reject"* does not mean *"accept"*. We are now ready to run our first real test.

***
## 3.6 The chi-square test for proportions (the right tool for A/B tests on clicks)

Click-through-rate A/B tests have an outcome that is fundamentally **yes/no**: every visitor either clicked or did not. The right hypothesis test for *"is the proportion of yes-es different between groups A and B?"* is the **chi-square test of independence**.

> **Callback to §2.6.6.** Last week we asked, informally, *"is light condition independent of urban/rural?"* and computed the joint vs marginal probabilities to compare. The chi-square test is the **formal version of that question** — *"are these two categorical variables (variant ID × clicked) independent?"*. When the null is true, the answer is *"yes, independent"* — variant has no effect on click. Rejecting the null says *"no, they are not independent"* — variant *does* affect the click rate.

### The 2×2 contingency table

For a 2-variant A/B test on clicks, we build a small table:

| | Clicked | Did not click | Row total |
|---|---:|---:|---:|
| Variant A | $a$ | $n_A - a$ | $n_A$ |
| Variant B | $b$ | $n_B - b$ | $n_B$ |
| **Column total** | $a + b$ | $(n_A + n_B) - (a+b)$ | $n_A + n_B$ |

Reading the symbols:

- $n_A$, $n_B$ — total impressions of variants A and B.
- $a$, $b$ — number of clicks (yes-es) on A and B.
- The interior cells are the **observed counts**.

### The chi-square statistic

Under the null *"variant and click are independent"*, the expected count in cell $(i, j)$ is

$$E_{ij} = \frac{(\text{row}_i \text{ total}) \times (\text{column}_j \text{ total})}{\text{grand total}}$$

The chi-square statistic measures how far the observed counts depart from these expected counts:

$$\chi^2 = \sum_{i, j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

Reading every symbol:

- $\chi^2$ ("chi-squared") — the test statistic. The name is just the Greek letter chi, squared.
- $O_{ij}$ — the *observed* count in cell $(i, j)$.
- $E_{ij}$ — the *expected* count in cell $(i, j)$ under the null.
- $\sum_{i,j}$ — sum over all four cells of the 2×2 table.

Intuitively: each term $(O - E)^2 / E$ measures how much a single cell departs from what the null predicts, scaled by the size of the expected count. We sum across all cells. The bigger $\chi^2$, the more the data departs from the null.

Under the null, this statistic has a **chi-square distribution with 1 degree of freedom** (for a 2×2 table). The p-value is the probability, under that distribution, of seeing a $\chi^2$ value at least as large as the one we observed.

### Tiny example by hand

Imagine A: 20 clicks of 1,000 impressions; B: 40 clicks of 1,000 impressions.

| | Clicked | Did not click | Row total |
|---|---:|---:|---:|
| A | 20 | 980 | 1000 |
| B | 40 | 960 | 1000 |
| **Total** | 60 | 1940 | 2000 |

Under independence, each cell expected count is $(\text{row}) \times (\text{col}) / 2000$:

- $E_{A, click} = 1000 \times 60 / 2000 = 30$
- $E_{A, noclick} = 1000 \times 1940 / 2000 = 970$
- $E_{B, click} = 30$, $E_{B, noclick} = 970$ (by symmetry)

So $\chi^2 = (20-30)^2/30 + (980-970)^2/970 + (40-30)^2/30 + (960-970)^2/970 \approx 3.33 + 0.10 + 3.33 + 0.10 \approx 6.87$. Looking that value up in a $\chi^2_{(df=1)}$ table gives $p \approx 0.009$ — the difference between 20/1000 and 40/1000 is unlikely under the null. We would reject.

### Computing it on a real Upworthy test

We will pick a specific 2-variant test from the safe window. The test `541083f71a` has two near-identical headlines that differ in *exactly one phrase* — the perfect way to isolate the effect of wording. Let us pull it out and look at it.

In [ ]:
# Pull out the one specific A/B test we will analyse.
test_id = "541083f71aec80dca6000092"

# Filter the DataFrame to just that test's rows, and keep only the columns we need.
chosen_test = df[df["clickability_test_id"] == test_id][["headline", "impressions", "clicks"]].reset_index(drop=True)

# Show what we have.
chosen_test

You should see two rows. Variant A has the headline *"This Made Me Want To Run Screaming Out Of My House"* with about 30 clicks on 5,200 impressions. Variant B has the headline *"This Made Me Run Screaming Out Of My House"* — same sentence minus "Want To" — with about 60 clicks on 5,100 impressions. **Twice the clicks, almost the same number of impressions.**

Let us compute the chi-square test.

In [ ]:
# Pull the (clicks, impressions) for variant A (the row indexed 0).
a_clicks, a_imps = int(chosen_test.loc[0, "clicks"]), int(chosen_test.loc[0, "impressions"])

# Pull the (clicks, impressions) for variant B (the row indexed 1).
b_clicks, b_imps = int(chosen_test.loc[1, "clicks"]), int(chosen_test.loc[1, "impressions"])

# Compute each variant's click-through rate as a fraction (clicks / impressions).
a_ctr = a_clicks / a_imps
b_ctr = b_clicks / b_imps

# Build the 2x2 contingency table: rows = variant, columns = clicked / did not click.
# Format: [[A_clicked, A_not_clicked], [B_clicked, B_not_clicked]].
contingency = np.array([
    [a_clicks, a_imps - a_clicks],
    [b_clicks, b_imps - b_clicks],
])

# Run the chi-square test via scipy. The function returns: chi-square statistic, p-value, degrees of freedom, expected counts.
chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency)

# Print everything for the worked example.
print(f"Variant A  (with 'Want To'):  {a_clicks} clicks / {a_imps} impressions  ->  CTR = {a_ctr*100:.2f}%")
print(f"Variant B  (without 'Want To'): {b_clicks} clicks / {b_imps} impressions  ->  CTR = {b_ctr*100:.2f}%")
print()
print(f"Observed difference (B - A) in CTR : {(b_ctr - a_ctr)*100:+.2f} percentage points")
print(f"Relative lift                       : {(b_ctr / a_ctr - 1) * 100:+.0f}%")
print()
print(f"Chi-square statistic : {chi2_stat:.3f}")
print(f"Degrees of freedom   : {dof}")
print(f"p-value              : {p_value:.5f}")
print()
print("Expected counts under H_0 (independence):")
print(pd.DataFrame(expected, columns=["Clicked", "Did not click"], index=["Variant A", "Variant B"]).round(1))

You should see approximately:

```
Variant A  (with 'Want To'):  30 clicks / 5203 impressions  ->  CTR = 0.58%
Variant B  (without 'Want To'): 60 clicks / 5113 impressions  ->  CTR = 1.17%

Observed difference (B - A) in CTR : +0.60 percentage points
Relative lift                       : +104%

Chi-square statistic : 9.945
Degrees of freedom   : 1
p-value              : 0.00161
```

**Reading these numbers honestly:**

- The **observed difference** is 0.60 percentage points. The *relative lift* is roughly +100% — variant B got about twice as many clicks per impression as variant A.
- The **p-value is 0.00161** — meaning that, *if the two headlines truly had the same click-through rate*, we would see a chi-square value this extreme (or more) about 1.6 times in 1,000 repetitions. That is unusual enough that we **reject the null** at the conventional $\alpha = 0.05$ level (and even at $\alpha = 0.01$).
- *In English for the memo:* removing "Want To" from this headline approximately doubled the click-through rate. The effect is too large to plausibly attribute to chance alone.

> **Why is variant B's headline more clickable?** That is an editorial question, not a statistical one. The data shows the effect; explaining *why* is a separate exercise. (My speculation: "want to" softens the action; the editted version is more visceral. But that is speculation.)

### A small detour — *lift* and *relative lift*, formally

The code above printed **two** flavours of *"how much better was B?"*. They are not the same number, and the difference matters for the memo. Let us slow down and read the formulas line by line.

#### Absolute lift — the gap in percentage points

The **absolute lift** is simply how many percentage points separate the two CTRs:

$$\text{absolute lift} \;=\; \hat{p}_B - \hat{p}_A$$

For our test:

$$\hat{p}_B - \hat{p}_A \;=\; 1.17\% - 0.58\% \;=\; +0.60 \text{ pp}$$

That is the `+0.60 percentage points` printed above. Absolute lift is in **percentage points** (pp), *not* in *percent* — a distinction stakeholders confuse constantly. Always write **pp** when you mean a difference between two rates.

#### Relative lift — the gap as a fraction of the baseline

The **relative lift** answers a different question: *"how much bigger is B's CTR **as a fraction of A's CTR**?"* That fraction is then expressed as a percentage for readability:

$$\text{relative lift} \;=\; \left(\frac{\hat{p}_B}{\hat{p}_A} - 1\right) \times 100\% \;=\; \frac{\hat{p}_B - \hat{p}_A}{\hat{p}_A} \times 100\%$$

The two right-hand sides are algebraically identical. The second form makes the meaning unmistakable: *"the gap between B and A, divided by the baseline A."*

Walking through the **first** form left-to-right with our numbers:

1. **`b_ctr / a_ctr`** — the **ratio** of B's CTR to A's CTR: $\;0.01174 \,/\, 0.00577 \;\approx\; 2.035$. *"B is about 2.035 times A."*
2. **`- 1`** — strip away the part of B that simply *matches* A; what is left is the *extra*, expressed as a fraction of A: $\;2.035 - 1 = 1.035$.
3. **`* 100`** — turn the fraction into a percentage humans read naturally: $\;1.035 \times 100\% \approx 103.5\%$, rounded for display by `:+.0f%` to **+104%**.

So *"the relative lift is +104%"* literally means *"B's CTR is 104% higher than A's CTR"* — equivalently, *"B's CTR is roughly 2.04 times A's CTR."*

#### Why the same data gives two very different-sounding numbers

The absolute lift (+0.60 pp) and the relative lift (+104%) describe the **same effect**. The relative number sounds dramatic because A's baseline is tiny: when the denominator $\hat{p}_A$ is small, even a small *absolute* gap balloons into a large *relative* gap. The reverse is also true — a +0.60 pp lift on top of a 50% baseline would be a relative lift of only $\;0.60\,/\,50 \times 100\% = \textbf{+1.2\%}$: same arithmetic, completely different business framing.

> **One subtle but important rule.** The baseline goes in the **denominator**. A is the baseline; B is the challenger. Computing the relative lift the other way around, $\left(\hat{p}_A \,/\, \hat{p}_B - 1\right) \times 100\%$, gives **−50.85%**, not −104% — you cannot lose more than 100% of something, but you can gain unboundedly. Always state which group is the baseline, and keep that group as the denominator.

For the memo, the two lifts answer two different leadership questions:

- **Absolute lift** — *"by how many percentage points did CTR move?"* Operationally meaningful for forecasting incremental clicks at fixed traffic.
- **Relative lift** — *"by what multiple did CTR move?"* This is the *"we doubled CTR!"* framing your stakeholders will quote.

An honest memo reports **both**, with the absolute lift first.

### Visualize the result


In [ ]:
# Create a side-by-side bar chart of the two click-through rates.
plt.figure(figsize=(7, 4))

# Bar heights are the CTRs of A and B as percentages.
plt.bar(["A (with 'Want To')", "B (without 'Want To')"], [a_ctr*100, b_ctr*100], edgecolor="black")

# Annotate each bar with its CTR value above it.
plt.text(0, a_ctr*100 + 0.05, f"{a_ctr*100:.2f}%", ha="center")
plt.text(1, b_ctr*100 + 0.05, f"{b_ctr*100:.2f}%", ha="center")

# Add the p-value as a subtitle so the test result is in the same picture.
plt.title(f"Upworthy test {test_id[:10]} — chi-square p = {p_value:.5f}")
plt.ylabel("Click-through rate (%)")
plt.ylim(0, max(a_ctr, b_ctr) * 100 * 1.3)
plt.tight_layout()
plt.show()

**What we see.** Two bars; the right-hand one is about twice as tall. The p-value of 0.00161 in the title tells the reader the difference is unlikely under the null.

> **Mini-recap of §3.6.** A 2-variant A/B test on clicks calls for a chi-square test on the 2×2 contingency table (variant × clicked). For Upworthy test `541083f71a` the result is $\chi^2 = 9.95$, $p = 0.00161$ — we reject the null. Variant B (*"This Made Me Run Screaming Out Of My House"*) wins, with a relative lift of about 100% over the version that begins *"This Made Me Want To Run..."*.

***
## 3.7 The two-sample t-test for comparing means

Not every A/B test has a yes/no outcome. Sometimes you want to compare **two groups on a continuous measurement** — average revenue per visitor, time spent on page, scroll depth, words read. For that you use the **two-sample t-test**.

We will keep this section short because the Upworthy data is yes/no (clicks), so chi-square is the right primary tool. But you will encounter t-tests constantly in growth work — they are the second-most-common A/B-test test.

### The setup

Two groups, $A$ and $B$, of sample sizes $n_A$ and $n_B$. We observe continuous values in each group. Compute:

- The **sample means** $\bar{x}_A$ and $\bar{x}_B$.
- The **sample standard deviations** $s_A$ and $s_B$.

> **Callback to §1.5 and §1.10.** The sample mean and sample standard deviation are exactly the objects we built in Session 01 §1.5. We are reusing them here. There is *nothing new* about $\bar{x}$ and $s$ — they are the same statistics; the t-test just combines them in a particular way.

### The t-statistic (Welch's version — the default in scipy)

$$t \;=\; \frac{\bar{x}_A - \bar{x}_B}{\sqrt{\,\dfrac{s_A^2}{n_A} + \dfrac{s_B^2}{n_B}\,}}$$

Reading every symbol:

- $t$ — the test statistic. The same letter as the *t-distribution* it follows under the null.
- $\bar{x}_A - \bar{x}_B$ — the **observed difference in sample means** (the numerator).
- $s_A^2 / n_A$ — the squared standard error of $\bar{x}_A$ (recall from §1.10: $\widehat{\operatorname{SE}}(\bar{x}) = s / \sqrt{n}$, so squared it is $s^2 / n$).
- $s_B^2 / n_B$ — the squared SE of $\bar{x}_B$.
- The denominator is the **standard error of the difference of means**.

In plain English: *"the t-statistic is the observed difference in means divided by the standard error of that difference"*. If the difference is many standard errors away from zero, the t-statistic is large in absolute value, and the p-value is small.

The *Welch* variant allows the two groups to have unequal variances. The *classic* t-test pools the variances; this is fine when variances really are equal, but a bad idea otherwise. **`scipy.stats.ttest_ind(..., equal_var=False)` runs the Welch version, and that should be your default.**

### Under the null

If the two groups truly have the same mean, the t-statistic follows a **t-distribution** with a particular degrees-of-freedom value (computed by Welch's formula). The p-value is the two-tailed area beyond $|t|$ in that distribution. SciPy does this for you.

### A small synthetic example

Since our Upworthy data is yes/no, let us demonstrate the t-test on a small synthetic example: two groups of "seconds spent on page" measurements. We will draw the data from numpy, knowing the true means and standard deviations.

In [ ]:
# Create a numpy random generator (modern API: `default_rng`) so the synthetic example is reproducible.
rng = np.random.default_rng(42)

# Synthetic group A: 200 visitors, average 30 seconds on page, SD = 10.
seconds_A = rng.normal(loc=30, scale=10, size=200)

# Synthetic group B: 200 visitors, average 33 seconds on page, SD = 12 (slightly different variance).
seconds_B = rng.normal(loc=33, scale=12, size=200)

# Compute the two sample means by hand for transparency.
mean_A = seconds_A.mean()
mean_B = seconds_B.mean()

# Run Welch's two-sample t-test (equal_var=False is the Welch version, the safer default).
# We pass (B, A) so scipy computes (mean_B - mean_A) / SE — the t-statistic then carries the
# same sign as the (B - A) difference we print below, which keeps the output easy to read.
t_stat, t_p = stats.ttest_ind(seconds_B, seconds_A, equal_var=False)

# Print the numbers.
print(f"Group A: n = {len(seconds_A)}, mean = {mean_A:.2f}s, SD = {seconds_A.std(ddof=1):.2f}s")
print(f"Group B: n = {len(seconds_B)}, mean = {mean_B:.2f}s, SD = {seconds_B.std(ddof=1):.2f}s")
print()
print(f"Observed difference (B - A): {mean_B - mean_A:+.2f} seconds")
print()
print(f"Welch t-statistic : {t_stat:.3f}")
print(f"p-value           : {t_p:.5f}")

You should see something like:

```
Group A: n = 200, mean = 29.70s, SD = 8.82s
Group B: n = 200, mean = 33.24s, SD = 12.23s

Observed difference (B - A): +3.55 seconds

Welch t-statistic : 3.325
p-value           : 0.00098
```

Group B spent about 3.5 seconds more on the page on average. The t-statistic is about 3.3 — and that number is just the §3.7 formula read out loud: $t = (\bar{x}_B - \bar{x}_A) \,/\, \widehat{\operatorname{SE}}_{\text{diff}}$. A printed $t \approx 3.3$ **literally means** the observed difference is about 3.3 times its own standard error — i.e., 3.3 standard errors away from zero under the null. The p-value is about 0.001, well below any conventional $\alpha$, so we **reject the null** that the two groups have the same mean time-on-page.

### When to use which test

A small decision rule:

- **Outcome is yes/no (clicked, converted, signed up)** → **chi-square** on the 2×2 (or k×2) contingency table.
- **Outcome is continuous (time, dollars, scroll depth)** → **two-sample t-test** (Welch's by default).
- **Outcome is a count over a fixed window** (rentals per hour, customers per minute) → either a Poisson regression, or just a t-test if the counts are large enough that the CLT kicks in.

We will use chi-square as our primary tool for the rest of this notebook because the actual Upworthy data is yes/no.

> **Mini-recap of §3.7.** The two-sample t-test compares **means of two groups**; Welch's variant handles unequal variances and is the safer default. The t-statistic is *"observed difference of means divided by its standard error"* — built from the same sample mean and SE we have used since Session 01.

***
## 3.8 Confidence intervals — the parametric companion to bootstrap CIs

Reporting just a p-value is bad practice. *"The lift was statistically significant ($p = 0.002$)"* tells the reader that something is unlikely to be zero — but how big is the effect? How precise is the estimate? The **confidence interval (CI)** answers both at once.

> **Major callback to §2.11.** Last week we computed a **bootstrap CI** for $P(\text{severe} \mid \text{rural})$ by resampling the data 10,000 times and taking the 2.5th and 97.5th percentiles of the bootstrap distribution. That gave us *"the rural severe rate is between 0.291 and 0.301 with 95% confidence"*. The bootstrap method needed no assumptions about the distribution.

> The CI we will compute here is the **parametric** version — it comes from a formula rather than a simulation. When the assumptions hold (and for proportions they hold well for the sample sizes we have), the two CIs give very similar numbers. The bootstrap is more general; the parametric CI is faster to compute.

### CI for a difference of proportions

For an A/B test with proportions $\hat{p}_A = a / n_A$ and $\hat{p}_B = b / n_B$, the standard 95% CI for the difference $\hat{p}_B - \hat{p}_A$ is:

$$(\hat{p}_B - \hat{p}_A) \;\pm\; 1.96 \cdot \widehat{\operatorname{SE}}_{\text{diff}}$$

where the standard error of the difference is:

$$\widehat{\operatorname{SE}}_{\text{diff}} \;=\; \sqrt{\dfrac{\hat{p}_A (1 - \hat{p}_A)}{n_A} \;+\; \dfrac{\hat{p}_B (1 - \hat{p}_B)}{n_B}}$$

Reading the symbols:

- $\hat{p}_A$, $\hat{p}_B$ — sample proportions in groups A and B (hat = estimated from data).
- $n_A$, $n_B$ — sample sizes.
- $1.96$ — the $z$-multiplier for a two-sided 95% CI from the standard Normal distribution. (95% of the area under a Normal lies within $\pm 1.96$ standard deviations of the mean.)
- $\widehat{\operatorname{SE}}_{\text{diff}}$ — the estimated standard error of the difference. Same kind of object as the SE from §1.10, just adapted for the difference of two proportions.

### Computing the CI for our Upworthy test

In [ ]:
# Use the variant-A and variant-B counts we already pulled in section 3.6.
# Compute the standard error of the difference using the formula above.
se_diff = np.sqrt(
    a_ctr * (1 - a_ctr) / a_imps
  + b_ctr * (1 - b_ctr) / b_imps
)

# Observed difference in CTR.
diff = b_ctr - a_ctr

# z-multiplier for a two-sided 95% CI: 1.96 (this is the standard value).
z = 1.96

# Lower and upper CI bounds.
ci_low  = diff - z * se_diff
ci_high = diff + z * se_diff

# Print the result with diff and CI expressed in percentage points for readability.
print(f"Observed difference (B - A) : {diff*100:+.3f} percentage points")
print(f"95% CI                       : ({ci_low*100:+.3f} pp, {ci_high*100:+.3f} pp)")
print(f"SE of difference              : {se_diff*100:.4f} pp")

You should see approximately:

```
Observed difference (B - A) : +0.597 percentage points
95% CI                       : (+0.237 pp, +0.957 pp)
SE of difference              : 0.1836 pp
```

**Reading this honestly:**

- Variant B's CTR is **0.60 percentage points higher** than A's, with 95% confidence somewhere between **+0.24 pp and +0.96 pp**.
- The CI does **not cross zero** — consistent with the chi-square p-value being small. (This is the **CI/p-value duality** we mentioned in §3.5: a 95% CI excluding zero is equivalent to rejecting at $\alpha = 0.05$.)
- The interval is fairly wide as a fraction of the point estimate. The lower bound (+0.24 pp) corresponds to a much smaller relative lift than the point (+104%). **The data tells us there is a real effect, but the precise magnitude has a meaningful margin of error.** This nuance is what an honest memo conveys.

### Recommended reporting style

For a stakeholder memo, **lead with the CI, not the p-value**:

> *"Variant B (without 'Want To') achieved a click-through rate of 1.17%, vs 0.58% for variant A — an absolute lift of 0.60 percentage points (95% CI: 0.24–0.96 pp; chi-square $p = 0.002$). The lift is statistically robust and the lower bound of the CI suggests a meaningful relative improvement even on the conservative end."*

That sentence carries the effect, the uncertainty, and the test, in that order. The p-value is a footnote.

> **Mini-recap of §3.8.** Confidence intervals are the parametric companion to bootstrap CIs. For a difference of proportions they have a clean closed form, $\hat{p}_B - \hat{p}_A \pm 1.96 \cdot \widehat{\operatorname{SE}}_{\text{diff}}$. They should *lead* every A/B test write-up — the p-value is a supporting actor.

***
## 3.9 p-hacking and the multiple-testing problem

Now we put on the analyst's honesty hat. Hypothesis testing is a powerful framework — but it is also abusable, and the most common abuses lead to **false claims of victory**. This section names the abuses so we can avoid them.

### What is p-hacking?

**p-hacking** is any practice that inflates the apparent statistical significance of a finding without genuinely earning it. Common forms:

- **Trying many subgroupings.** *"The test was not significant overall, but it was significant for users on iOS, on Tuesdays, who came in from organic search."* If you try enough subsets, one of them will look significant by chance.
- **Stopping early when the result is significant.** *"We hit $p < 0.05$ at week 2, so we stopped the test and shipped B."* This inflates the false-positive rate dramatically — the test was designed assuming a fixed sample size, not adaptive stopping.
- **HARKing — Hypothesizing After Results are Known.** Looking at the data, finding a pattern, and then writing the hypothesis as if it had been pre-specified. The pre-registered version of the same finding would not have survived a multiple-testing correction.
- **Outcome switching.** Picking the metric that "worked" out of several you tracked, then reporting only that one.

The thread is the same in all of these: **the analyst quietly does multiple comparisons but reports only the winner, as if the winner had been the only thing examined**. The statistics break under this.

### The multiple-testing problem — let us see it directly

If you run **one** test under the null hypothesis (no real effect) at $\alpha = 0.05$, the probability of a false positive is, by definition, 5%. If you run **20 independent tests** under the null, the probability that at least one is significant by chance is

$$1 - (1 - 0.05)^{20} \approx 64\%$$

So in a quarter where your growth team runs 20 A/B tests on headlines that have **no real effect**, you should *expect* about one false-positive winner, and you should be unsurprised to see two or three. That false winner will then get shipped — site-wide — and someone will get credit for it.

### A direct demonstration on Upworthy

We will take all the 2-variant tests in our safe-window dataset (that have decent sample size), compute the chi-square p-value for each, and look at the **distribution of p-values across all those tests**. Under the null *"no headline really differs from any other"*, that distribution would be approximately uniform on $[0, 1]$. Bumps above the uniform line indicate real effects.

In [ ]:
# Filter to tests with exactly 2 variants — the simplest case for a chi-square test.
sizes = df.groupby("clickability_test_id").size()
two_variant_ids = sizes[sizes == 2].index

# For each 2-variant test with at least 1000 impressions on each side and at least 5 clicks on each side,
# compute the chi-square p-value. Collect them all in a list.
p_values = []
for tid, grp in df[df["clickability_test_id"].isin(two_variant_ids)].groupby("clickability_test_id"):
    pair = grp[["impressions", "clicks"]].values
    if len(pair) != 2:
        continue
    aI, aC = int(pair[0][0]), int(pair[0][1])
    bI, bC = int(pair[1][0]), int(pair[1][1])
    if aI < 1000 or bI < 1000 or aC < 5 or bC < 5:
        continue
    # 2x2 contingency table for this single test.
    table = np.array([[aC, aI - aC], [bC, bI - bC]])
    # Compute chi-square; we only want the p-value.
    _, p, _, _ = stats.chi2_contingency(table)
    p_values.append(p)

# Convert to a numpy array for easy summarising.
p_values = np.array(p_values)

# Report how many tests we ran, and how many were "significant" at conventional thresholds.
print(f"Number of two-variant tests analysed : {len(p_values):,}")
print(f"Number with p < 0.05                 : {(p_values < 0.05).sum():,}  ({(p_values < 0.05).mean()*100:.1f}%)")
print(f"Number with p < 0.01                 : {(p_values < 0.01).sum():,}  ({(p_values < 0.01).mean()*100:.1f}%)")
print(f"Number with p < 0.001                : {(p_values < 0.001).sum():,}  ({(p_values < 0.001).mean()*100:.1f}%)")
print()
print("Under a strict null (no real effects anywhere) we would expect:")
print(f"  ~5% with p < 0.05, ~1% with p < 0.01, ~0.1% with p < 0.001.")

You should see numbers like:

```
Number of two-variant tests analysed : 220
Number with p < 0.05                 :  46  (20.9%)
Number with p < 0.01                 :  17  (7.7%)
Number with p < 0.001                :   6  (2.7%)
```

**Reading this:** Upworthy headline tests *as a population* show more significant results than chance alone would produce (20% at p < 0.05 vs the 5% the null predicts; 7.7% at p < 0.01 vs the 1% predicted). So real effects *do* exist in this population. But:

- The rate of "significant" findings is still consistent with about a quarter of them being false positives that would have happened by chance.
- If your growth team uses *"p < 0.05"* as the shipping rule without any correction, you will ship false winners.

### A histogram of all 220 p-values

In [ ]:
# Plot a histogram of all the p-values; under the null they would be roughly uniform on [0, 1].
plt.figure(figsize=(10, 4))

# 20 bins gives a clean uniform-or-not picture.
plt.hist(p_values, bins=20, edgecolor="black")

# Reference line at 1/20 of the total count (what each bin would hold under a uniform distribution).
expected_per_bin = len(p_values) / 20
plt.axhline(expected_per_bin, color="red", linestyle="--",
            label=f"uniform under null (~{expected_per_bin:.1f} per bin)")

# Title and labels.
plt.title("Distribution of p-values across 220 Upworthy A/B tests")
plt.xlabel("p-value")
plt.ylabel("number of tests")
plt.legend()
plt.tight_layout()
plt.show()

**What we see.** The leftmost bins (small p-values, evidence of real effects) sit clearly above the uniform-null line — there *are* real differences among Upworthy headlines, not just chance. But the bins toward $p = 1$ (no effect) are also populated; many headline tests really are inconclusive. The histogram is informative in itself: *the population of A/B tests is a mix of true effects and null effects.*

### Two ways to correct for multiple testing

When you run many tests at once, you have two practical options:

1. **Bonferroni correction.** Divide your significance threshold by the number of tests: if you ran 20 tests and want family-wise $\alpha = 0.05$, declare each individual test significant only if its $p < 0.05 / 20 = 0.0025$. Simple and conservative — it controls the probability of *any* false positive across all 20.

2. **Benjamini–Hochberg / False Discovery Rate (FDR).** Sort all the p-values from smallest to largest; declare significant the largest set such that the expected fraction of false discoveries among them stays below a target (typically 5%). Less conservative than Bonferroni, more appropriate when you genuinely expect many real effects.

We will not implement either by hand in this notebook (Session 03 is already dense). The point is the awareness: **report all the tests you ran**, **pre-register hypotheses**, and **apply a multiple-testing correction whenever you analyse more than one test at a time**. Single A/B tests reported in isolation do not need correction; quarterly reports across 30 experiments absolutely do.

> **Mini-recap of §3.9.** When you run many tests under the null, false positives are statistically guaranteed. The fix is **pre-registration**, **honest reporting of every test you ran**, and **multiple-testing correction** (Bonferroni or FDR). Across our 220 Upworthy 2-variant tests, ~21% reached $p < 0.05$ — consistent with a mix of true and null effects.

***
## 3.10 Statistical power — why "no significant difference" is not the same as "no difference"

The opposite of p-hacking is **silent under-powering** — running a test that *could not have detected an effect even if one were there*, and then reporting *"no significant difference"* as if the experiment had been informative. It looks responsible. It is often misleading.

### What is statistical power?

**Power** is the probability that your test correctly rejects $H_0$ *when the alternative is true*:

$$\text{Power} = P(\text{reject } H_0 \mid H_1 \text{ is true with effect size } \delta)$$

In English: *"if there really is an effect of size $\delta$, how often will my test pick it up?"*. Power depends on three things:

- The **effect size** $\delta$ — bigger effects are easier to detect.
- The **sample size** $n$ — more data, more power.
- The **significance level** $\alpha$ — a stricter $\alpha$ (e.g., 0.01 vs 0.05) reduces power but also reduces false positives.

The conventional target is **80% power**: design your test so that, *if the effect is real and at least as big as you care about*, you will detect it 80% of the time.

### A back-of-envelope intuition

For a 2-sample test of proportions with both groups at $n = 5{,}000$, you can detect a 0.5 percentage-point lift over a 1% baseline with reasonable power. To detect a 0.1 percentage-point lift over the same baseline, you would need each group to be roughly **25 times** as big — into the hundreds of thousands of impressions per arm. The $\sqrt{n}$ pattern from §1.10 and §2.11 returns: halving the detectable effect requires roughly *four times* the sample size.

### A concrete consequence: the Upworthy null result

Some Upworthy tests have very large sample sizes — over 20,000 impressions total — and *still* return non-significant p-values. Let us look at one.

In [ ]:
# Find the highest-sample-size 2-variant test that has a non-significant result.
# We will use one identified during our exploration earlier.
null_test_id = "5147c10da53b840002004989"
null_test = df[df["clickability_test_id"] == null_test_id][["headline", "impressions", "clicks"]].reset_index(drop=True)

# Pull the two variants' counts.
nA_clicks, nA_imps = int(null_test.loc[0, "clicks"]), int(null_test.loc[0, "impressions"])
nB_clicks, nB_imps = int(null_test.loc[1, "clicks"]), int(null_test.loc[1, "impressions"])

# Compute their CTRs.
nA_ctr = nA_clicks / nA_imps
nB_ctr = nB_clicks / nB_imps

# Chi-square test.
table = np.array([[nA_clicks, nA_imps - nA_clicks], [nB_clicks, nB_imps - nB_clicks]])
null_chi2, null_p, _, _ = stats.chi2_contingency(table)

# Print everything.
print("Headlines:")
print(f"  A: {null_test.loc[0, 'headline']}")
print(f"  B: {null_test.loc[1, 'headline']}")
print()
print(f"A: {nA_clicks} clicks / {nA_imps} impressions  ({nA_ctr*100:.2f}%)")
print(f"B: {nB_clicks} clicks / {nB_imps} impressions  ({nB_ctr*100:.2f}%)")
print()
print(f"Observed difference: {(nB_ctr - nA_ctr)*100:+.3f} pp")
print(f"Chi-square p-value : {null_p:.4f}")

You should see something like:

```
Headlines:
  A: A Group Of Gamers May Have Unlocked The Cure To AIDS
  B: Did AIDS Just Get Cured By A Bunch Of Gamers?

A: 78 clicks / 11192 impressions  (0.70%)
B: 85 clicks / 10942 impressions  (0.78%)

Observed difference: +0.080 pp
Chi-square p-value : 0.5376
```

**Reading this honestly:** even with **22,000 total impressions**, the test cannot distinguish 0.70% from 0.78%. The p-value of 0.54 means *"if there were no real difference, we would see a gap this big about half the time by chance"*. We **fail to reject** the null.

The correct memo language is **not** *"Headlines A and B are the same"*. It is:

> *"With 11,000 impressions per variant we could not detect a meaningful difference between A and B (chi-square $p = 0.54$, 95% CI for difference: −0.15 to +0.31 pp). The data is consistent with either headline being slightly better than the other by up to about 0.3 percentage points. If a difference smaller than that matters to the business, we would need substantially more impressions to detect it."*

That sentence states what the data does and does not support. The right kind of honest.

> **Mini-recap of §3.10.** Power is the probability of detecting a real effect of given size. Underpowered tests systematically miss true effects. *"No significant difference"* should never appear in a memo without an accompanying statement of *"what we could and could not have detected at this sample size"*. State the minimum detectable effect, not just the result.

***
## 3.11 Practical vs statistical significance

The most important conceptual distinction in this whole notebook. Read it carefully.

- **Statistical significance** is a property of the *test*. It answers: *"is this effect distinguishable from zero given my sample size?"*
- **Practical significance** is a property of the *business*. It answers: *"is this effect big enough to matter?"*

**They are completely different questions.** A finding can be statistically significant but practically trivial (with a huge sample, you can detect a 0.001 percentage-point lift). Conversely, a finding can be practically huge but statistically inconclusive (a 30% lift on a small sample with wide CIs).

### A worked counter-example — what happens with a huge sample

Imagine we ran the Upworthy test in §3.6, but at **100× the impressions** — about 500,000 per variant — and the CTRs stayed at exactly 0.58% vs 0.59% (a difference of just 0.01 percentage points, which is a relative lift of about 2%). Let us see what the chi-square test does:

In [ ]:
# Hypothetical: 500,000 impressions per variant, CTRs of 0.58% and 0.59%.
hyp_nA, hyp_nB = 500_000, 500_000

# Clicks consistent with those CTRs (rounded to whole numbers).
hyp_A_clicks = int(round(0.0058 * hyp_nA))
hyp_B_clicks = int(round(0.0059 * hyp_nB))

# Build the contingency table.
hyp_table = np.array([
    [hyp_A_clicks, hyp_nA - hyp_A_clicks],
    [hyp_B_clicks, hyp_nB - hyp_B_clicks],
])

# Chi-square test on the hypothetical scenario.
hyp_chi2, hyp_p, _, _ = stats.chi2_contingency(hyp_table)

# Print the result.
print(f"Hypothetical: variant A CTR = {hyp_A_clicks/hyp_nA*100:.3f}%, variant B CTR = {hyp_B_clicks/hyp_nB*100:.3f}%")
print(f"Absolute difference          : {(hyp_B_clicks/hyp_nB - hyp_A_clicks/hyp_nA)*100:+.3f} pp")
print(f"Relative lift                 : {(hyp_B_clicks/hyp_nB) / (hyp_A_clicks/hyp_nA) * 100 - 100:+.1f}%")
print(f"Chi-square statistic          : {hyp_chi2:.3f}")
print(f"p-value                       : {hyp_p:.5f}")

You should see something like:

```
Hypothetical: variant A CTR = 0.580%, variant B CTR = 0.590%
Absolute difference          : +0.010 pp
Relative lift                 : +1.7%
Chi-square statistic          : 4.31
p-value                       : 0.03781
```

**Reading this:** a 0.01 percentage-point absolute lift (or 1.7% relative) is **statistically significant** at $p = 0.038$, just barely below the 0.05 threshold. By the conventional rule, you would *"reject the null"* and *"declare a winner"*.

**Should you ship it?** Almost certainly not. A 0.01 percentage-point lift is a rounding error from the perspective of an editorial team writing headlines. The effort to maintain a separate winning headline plus the cost of running the test continuously would dwarf the gain. *"Statistically significant"* is not the same as *"worth doing"*.

### The two questions every A/B test memo must answer

When you write the stakeholder memo, lead with answers to **both** of these:

1. **"Is the effect bigger than zero?"** This is the statistical question — answered by the CI and the p-value. (For our real Upworthy test in §3.6: yes, the CI excludes zero.)
2. **"Is the effect big enough to matter?"** This is the business question — answered by the effect size (point estimate and CI) compared with a pre-specified minimum important effect. (For our real Upworthy test: yes, the relative lift is +100% and the lower CI bound is still a meaningful improvement.)

If the answer to (1) is *yes* and to (2) is also *yes*: ship.
If (1) is *yes* and (2) is *no*: do not ship — the effect is real but too small to be worth the operational cost.
If (1) is *no*: do not ship — the data does not yet support a winner.

**Most A/B test write-ups in industry stop at question (1) and call it a victory.** That is the trap.

> **Mini-recap of §3.11.** Statistical significance answers *"can we tell this is not zero?"*. Practical significance answers *"is this big enough to matter?"*. Both must appear in every honest A/B test memo. With a large enough sample, even trivially small effects become statistically significant — the p-value alone is a poor decision criterion.

***
## 3.12 Our API calls — Anthropic tool use for the A/B test analysis plan

> **Callback to §1.11 and §2.13.** In Session 01 we made our first Anthropic API call — plain text. In Session 02 we asked Anthropic for a JSON outline and *validated* its structure with simple Python checks **after** the call. This week we step up to **schema-enforced** structured output — using **Anthropic's tool use** feature, which forces the model to return JSON matching a schema we define in Python **at generation time**, not just validated afterwards. Same provider, a stronger guarantee.

We will make **two API calls** this week, both to Anthropic:

1. **Call 1 — schema-enforced plan.** Anthropic generates an **A/B test analysis plan** as a JSON object that *conforms to a schema we define in Python*. The schema specifies the fields, their types, and which fields are required. Anthropic's tool-use feature guarantees the response will match the schema — the model literally cannot reply with malformed or off-schema output.
2. **Call 2 — language work.** Anthropic drafts the **stakeholder memo paragraph** from the numbers we have already computed. Same *"Python computes, model interprets"* discipline as before — the model never invents numbers.

This is a single-provider workflow: one SDK, one API key, two different *modes* of using the model — structural work and language work.

### How Anthropic tool use enforces a schema

The trick is to wrap the request as a **tool call**. We define a "tool" with a name, a description, and a JSON Schema for its **input**. Then we tell Anthropic the model **must** call this tool. The model now has only one valid move: emit a tool call whose arguments match our schema. Anthropic enforces that match before returning the response — non-conforming output is impossible.

Pydantic gives us the schema for free. We write a regular Python class; `.model_json_schema()` returns the JSON Schema dict Anthropic expects. The same Pydantic class lets us re-wrap the validated response for type-safe access in the rest of the notebook.

### Step 1 — verify the API key is available

In [ ]:
# Import os so we can read environment variables.
import os

# Read the Anthropic API key from the environment.
anth_key = os.environ.get("ANTHROPIC_API_KEY")

# If it is missing, halt the notebook with a clear, actionable message.
if not anth_key:
    raise SystemExit(
        "ANTHROPIC_API_KEY is not set in your environment.\n"
        "Follow Step 4 of the repository README, close VS Code, reopen, and re-run this cell."
    )

# Confirm the key is set without printing its value.
print(f"ANTHROPIC_API_KEY is set. Key length: {len(anth_key)} characters.")

### Step 2 — create the Anthropic client

In [ ]:
# Import the official Anthropic Python SDK.
import anthropic

# Create the Anthropic client; it picks up ANTHROPIC_API_KEY from the environment automatically.
client = anthropic.Anthropic()

# Print a confirmation that the client object was created successfully.
print("Anthropic client ready.")

### Step 3 — define a Pydantic schema for the A/B test analysis plan

Anthropic's tool-use feature takes a **JSON Schema** at request time and *forces* the model's tool call to match it. The cleanest way to specify a schema in Python is via a **Pydantic model** — a regular Python class that doubles as a schema definition. We will write the class in Python, ask Pydantic for the corresponding JSON Schema dict, and hand that dict to Anthropic as the tool's `input_schema`.

The manager's brief asked for four sections, but a real pre-registration spec needs more precision than that. We expand the brief into **six required string fields** — the pieces a statistician would need to reproduce the analysis without further conversation: metric definition, the two hypotheses, the test choice with its assumptions, the significance level and decision rule, and a one-line note on multiple-testing handling.

In [ ]:
# Import Pydantic's BaseModel and Field helpers. Pydantic is installed in the `ailab` venv as a dependency of the anthropic SDK.
from pydantic import BaseModel, Field

# Define the structure of an A/B test analysis plan as a Pydantic model.
# Each field has a type (str) and a `description` that the model will read when filling the field.
class ABTestAnalysisPlan(BaseModel):
    metric_definition: str = Field(
        description="Exactly what we are measuring. One sentence. Include the unit of analysis (per visitor, per impression, per session)."
    )
    null_hypothesis: str = Field(
        description="The H_0 statement, in plain English, for a 2-variant A/B test on click-through rate."
    )
    alternative_hypothesis: str = Field(
        description="The H_1 statement, in plain English. Specify two-sided or one-sided."
    )
    test_choice_and_assumptions: str = Field(
        description="Which statistical test we use, why, and what assumptions it requires."
    )
    significance_level_and_decision_rule: str = Field(
        description="The pre-committed alpha and the rule for declaring a winner."
    )
    multiple_testing_note: str = Field(
        description="One sentence on how multiple-testing risk is handled (Bonferroni, FDR, or 'this is a single pre-registered test')."
    )

# Print confirmation.
print("Pydantic schema ABTestAnalysisPlan defined.")

### Step 4 — call Anthropic with a forced tool

We build a tool spec from our Pydantic schema and tell Anthropic the model **must** call it. Two pieces matter:

- `tools=[plan_tool]` — declares the one tool available to the model. The `input_schema` is our Pydantic-derived JSON Schema, so any tool call the model emits must match the schema exactly.
- `tool_choice={"type": "tool", "name": "submit_analysis_plan"}` — forces the model to call this specific tool. Without this, the model is free to reply with plain text; with it, the only valid move is a schema-conforming tool call.

The response comes back as a list of content blocks. We scan for the `tool_use` block — that block's `input` is the validated dictionary, ready to be wrapped back into our Pydantic class for type-safe access.

In [ ]:
# Build the Anthropic tool spec from our Pydantic schema.
# `model_json_schema()` returns a JSON Schema dict — exactly what Anthropic's `input_schema` field expects.
plan_tool = {
    "name": "submit_analysis_plan",
    "description": "Submit a complete A/B test analysis plan. You MUST call this tool with every required field filled in.",
    "input_schema": ABTestAnalysisPlan.model_json_schema(),
}

# Call Claude. `tool_choice` forces the model to call this specific tool — it cannot reply with plain text.
plan_response = client.messages.create(
    model="claude-haiku-4-5",                                       # same Haiku model used in §1.11 and §2.13
    max_tokens=1024,                                                # generous cap; the tool input is short
    system=(                                                        # Experiment Coach persona
        "You are an Experiment Coach for a digital media growth team. "
        "You enforce structural rigour before any A/B test is analysed. "
        "When asked for an analysis plan, fill in every required field with a one-sentence, plain-English answer. "
        "You never invent numbers; you focus on framing, not findings."
    ),
    tools=[plan_tool],                                              # the only tool the model is allowed to call
    tool_choice={"type": "tool", "name": "submit_analysis_plan"},   # force this exact tool — no free-text reply allowed
    messages=[
        {
            "role": "user",
            "content": (
                "I am about to analyse a 2-variant A/B test of headlines on the Upworthy archive. "
                "The metric is click-through rate (clicks / impressions), the unit of analysis is the visitor's first impression. "
                "Please fill in the analysis plan. Use a 2-sided alternative and pre-commit to alpha = 0.05. "
                "The test is a single pre-registered comparison, so no multiple-testing correction is needed."
            )
        }
    ],
)

# Scan the response for the tool_use block. Because tool_choice forced our tool, exactly one such block exists.
plan_dict = None
for block in plan_response.content:
    if block.type == "tool_use":
        plan_dict = block.input    # this dict already conforms to our schema — Anthropic enforced it
        break

# Wrap the dict back into our Pydantic class for type-safe, attribute-style access in the rest of the notebook.
plan = ABTestAnalysisPlan(**plan_dict)

# Show the plan field by field for easy reading.
for field_name, value in plan.model_dump().items():
    print(f"-- {field_name} --")
    print(f"  {value}\n")

**What we see.** Anthropic returns a tool call whose `input` is a JSON object with exactly the six fields our Pydantic schema specified, each filled with a one-sentence statement. The `tool_choice` guarantee means no validation step is needed afterwards — the model literally cannot return something that fails the schema. This is genuinely stronger than our Session 02 approach (free-form JSON + Python-side validation), because schema conformance is enforced *during generation*, not checked after the fact.

### Call 2 — Anthropic drafts the stakeholder memo paragraph

Now we hand our **computed numbers** to Anthropic and ask it to write the "headline findings" paragraph for the memo. Same *"Python computes, model interprets"* rule as Sessions 01 and 02 — Anthropic does the language work, never the math.

In [ ]:
# Build a tight numerical summary of the worked Upworthy test for Anthropic to translate into prose.
findings_summary = (
    f"Dataset: Upworthy Research Archive (CC BY 4.0), confirmatory subset, randomization-issue window excluded.\n\n"
    f"Test analysed: clickability_test_id {test_id[:10]}.\n\n"
    f"Variant A headline: {chosen_test.loc[0, 'headline']}\n"
    f"  A: {a_clicks} clicks / {a_imps} impressions  ->  CTR = {a_ctr*100:.2f}%\n\n"
    f"Variant B headline: {chosen_test.loc[1, 'headline']}\n"
    f"  B: {b_clicks} clicks / {b_imps} impressions  ->  CTR = {b_ctr*100:.2f}%\n\n"
    f"Observed CTR difference (B - A): {diff*100:+.2f} percentage points.\n"
    f"95% CI for the difference: ({ci_low*100:+.2f} pp, {ci_high*100:+.2f} pp).\n"
    f"Chi-square test: chi2 = {chi2_stat:.2f}, df = 1, p = {p_value:.5f}.\n"
    f"Relative lift: about {(b_ctr / a_ctr - 1) * 100:.0f}%."
)

# Print what we are sending so the participant can see the inputs.
print("Numbers we are sending to Anthropic:\n")
print(findings_summary)

In [ ]:
# Call Anthropic to write the stakeholder paragraph for the memo.
memo_response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=500,
    system=(
        "You translate analysis numbers into one short paragraph for a digital media growth team's stakeholder memo. "
        "Lead with the effect size (absolute and relative). Mention the 95% confidence interval before the p-value. "
        "Use plain English. No jargon. No mathematical symbols. "
        "Use ONLY the numbers given to you; never invent new numbers."
    ),
    messages=[
        {
            "role": "user",
            "content": (
                "Here are the numbers from our A/B test of two headlines on the Upworthy archive:\n\n"
                + findings_summary
                + "\n\nWrite ONE short paragraph (4 to 6 sentences) suitable for the 'Headline findings' "
                + "section of a stakeholder memo. Lead with the effect, name the confidence interval, "
                + "and close with a one-sentence recommendation."
            )
        }
    ]
)

# Print Anthropic's paragraph.
print(memo_response.content[0].text)

**What we see.** A short, plain-English paragraph that leads with effect size and CI, mentions the p-value, and closes with a recommendation. The model **did not invent any numbers** — it was given the entire set it was allowed to use.

This is the two-call pattern Session 03 introduces, both to the same provider:

- **Anthropic tool use** for **schema-enforced structure** (the analysis plan).
- **Anthropic plain text** for **language work on already-computed numbers** (the memo paragraph).

The point is *not* that Anthropic is the only provider worth using — there are other capable providers and you may encounter them in your career. The point is that **schema-enforced structure is a feature of the Messages API itself** via tool use; you do not need a second provider, a second SDK, or a second API key to get a guarantee that the model's output conforms to the shape your code expects.

> **Mini-recap of §3.12.** Two API calls, one provider. Anthropic tool use with a Pydantic-derived schema **guarantees** structured output at generation time. Plain-text Anthropic remains our default for language work with computed numbers. Same *"Python computes, model interprets"* discipline as Sessions 01 and 02.

***
## 3.13 The A/B test analysis plan and stakeholder memo — fully worked

It is Friday afternoon. Here are the two artifacts you would hand the growth team manager. Both are grounded entirely in numbers computed in this notebook.

---

### Artifact 1 — A/B test analysis plan

**Test ID:** `541083f71aec80dca6000092`

**Variants:**
- A: *"This Made Me Want To Run Screaming Out Of My House"*
- B: *"This Made Me Run Screaming Out Of My House"*

**Metric definition.** Click-through rate (CTR) = clicks divided by impressions. The unit of analysis is a single visitor's first impression of one variant. Each visitor sees exactly one variant; no visitor contributes to both groups.

**Null hypothesis $H_0$.** The two headlines have the same true click-through rate.

**Alternative hypothesis $H_1$.** The two headlines have **different** click-through rates (two-sided).

**Test and assumptions.** Pearson chi-square test of independence on a 2×2 contingency table (variant × clicked/not). Assumes (i) each impression is an independent randomized assignment to a variant, and (ii) expected cell counts are large enough that the chi-square approximation is accurate (rule of thumb: all $E_{ij} > 5$). For our test, the smallest expected count is ~45, so the approximation is comfortable.

**Significance level $\alpha = 0.05$**, pre-committed before viewing data. Decision rule: reject $H_0$ if $p < \alpha$.

**Multiple-testing handling.** This is a **single pre-registered test**, not a fishing expedition. No correction is required. *If* the growth team aggregates findings across multiple concurrent tests at quarterly review, Bonferroni or FDR correction would be applied at that aggregation stage.

**Data caveats.** The randomization-issue window (2013-06-25 to 2014-01-10) has been excluded per the archive maintainers' guidance.

---

### Artifact 2 — Stakeholder memo

**To:** Growth Team Lead
**From:** \[Your name\], Junior Analyst
**Re:** A/B test recommendation — Upworthy headline test `541083f71a`
**Date:** Friday, end of week 3

#### 1. What we tested

Two near-identical variants of the same headline, differing only in the phrase "Want To":

- A: *"This Made Me **Want To** Run Screaming Out Of My House"*
- B: *"This Made Me Run Screaming Out Of My House"*

Each variant received roughly 5,100 impressions during the test window.

#### 2. Headline findings

- Variant B (without "Want To") achieved a **click-through rate of 1.17%**, vs **0.58% for variant A** — an absolute lift of **+0.60 percentage points** (95% CI: +0.24 to +0.96 pp).
- The relative lift is approximately **+100%** — variant B drew **about twice the clicks per impression** of variant A.
- The chi-square test rejects the null of equal CTRs at $\chi^2 = 9.95$ on 1 d.f., **$p = 0.002$**. The 95% CI excludes zero; the effect is **statistically robust**.
- **Practical significance:** even on the conservative lower bound of the CI (+0.24 pp), a relative lift of roughly +40% over baseline is comfortably above the team's typical 10%-relative threshold for shipping. The effect is **practically meaningful** as well as statistically significant.

#### 3. Limitations

- This is one test of one headline pair. **Headline writing is highly context-dependent**: the same edit might do nothing for a different article.
- We have **about 10,300 total impressions** here (~5,100 per variant), but no per-segment analysis. We cannot say whether the lift is uniform across audience segments (mobile vs desktop, returning vs new, etc.); the data does not contain segment information.
- The dataset is from 2014. **User behaviour may have shifted since then**; treat this as evidence about *the linguistic pattern*, not as a directly applicable point estimate for today.
- The randomization-issue window was excluded per the archive maintainers' guidance; this test is from outside that window.

#### 4. Recommendation

**Ship variant B.** The effect is large in both absolute and relative terms, the confidence interval comfortably excludes zero on the conservative end, and the linguistic pattern (removing softening modifiers like "Want To") is generalizable enough to be worth carrying forward as a writing principle. Track post-rollout click-through rates for two weeks to confirm the gain holds in production.

#### 5. Open questions

- **Does the same edit work on other headlines?** Run a structured follow-up A/B test on three more headline pairs that contain "Want To" or similar softening modifiers.
- **Does CTR translate to deeper engagement?** A higher click-through rate does not automatically mean better long-term retention. Check time-on-page and return-rate in production.
- **Is the lift uniform across devices?** Sample-size constraints prevent us from telling now. A larger production rollout would.

---

That is your two-artifact deliverable. **Save your own memo as `_reports/session03_ab_test_memo.md`** in your course directory. The instructor will review it next week.

> **Mini-recap of §3.13.** Two artifacts: a pre-registration-style analysis plan (generated with Anthropic tool use, grounded in the rigour of §3.5–3.8) and a stakeholder memo (drafted with Anthropic on numbers we computed ourselves). Every claim traces back to a cell in this notebook. The structure prevents the most common A/B-test over-claiming mistakes — p-hacking through subgroups, ignoring practical significance, conflating "no significant difference" with "no difference".

***
## 3.14 References — what to study to deepen this session

A longer list than usual this week — hypothesis testing has a lot of moving parts. Pick **three or four** of the StatQuest videos plus the Khan Academy *Significance tests* unit. The single most important video is **Statistical Power**, because most analysts have never been taught it properly.

### StatQuest videos (YouTube)

| Video | What it clarifies |
|---|---|
| [Hypothesis Testing and the Null Hypothesis](https://www.youtube.com/watch?v=0oc49DyA3hU) | The framework from §3.5 — the "we don't believe X and try to reject it" framing, told visually. |
| [Alternative Hypotheses: Main Ideas](https://www.youtube.com/watch?v=5koKb5B_YWo) | One-sided vs two-sided $H_1$, when each is appropriate. |
| [p-values: What they are and how to interpret them](https://www.youtube.com/watch?v=vemZtEM63GY) | The correct interpretation of the p-value — and the four most common misinterpretations. Must-watch. |
| [How to calculate p-values](https://www.youtube.com/watch?v=JQc3yx0-Q9E) | Derives a p-value step-by-step from a sampling distribution. Pairs with §3.5's CLT callback. |
| [StatQuickie: Which t test to use](https://www.youtube.com/watch?v=NkGvw18zlGQ) | Two minutes on choosing between one-sample, paired, and two-sample t-tests; Welch awareness included. |
| [Using Linear Models for t-tests and ANOVA](https://www.youtube.com/watch?v=NF5_btOaCig) | Shows that the t-test is a special case of linear regression — a useful bridge to Session 04. |
| [Confidence Intervals, Clearly Explained](https://www.youtube.com/watch?v=TqOeMYtOc1w) | The parametric CI from §3.8. Pairs with the bootstrap CI from §2.11. |
| [p-hacking: What it is and how to avoid it](https://www.youtube.com/watch?v=HDCOUXE3HMM) | Direct companion to §3.9. The single most important honesty video on the list. |
| [False Discovery Rates (FDR), Clearly Explained](https://www.youtube.com/watch?v=K8LQSvtjcEo) | The Benjamini–Hochberg correction in §3.9, visualized. Important for quarterly aggregations. |
| [Statistical Power, Clearly Explained](https://www.youtube.com/watch?v=Rsc5znwR5FA) | The §3.10 idea. **The single most important video on this list** if you have never been taught power properly. |
| [Power Analysis](https://www.youtube.com/watch?v=VX_M3tIyiYk) | Designing a test with adequate power *before* running it. Sample-size calculation intuition. |
| [Bootstrapping (Main Ideas)](https://www.youtube.com/watch?v=Xz0x-8-cgaQ) | Reinforces §2.11 — useful background for understanding what parametric tests are doing. |

### Khan Academy resources

- [Significance tests (hypothesis testing) unit](https://www.khanacademy.org/math/statistics-probability/significance-tests-one-sample) — the structured course; pair with the StatQuest p-value video for full coverage.
- [Z-statistics vs T-statistics](https://www.youtube.com/watch?v=5ABpqVSx33I) — when to use which, especially for small samples.
- [Introduction to t statistics](https://www.youtube.com/watch?v=hxZ6uooEJOk) — the t-statistic from §3.7, beginner-friendly.
- [Two-sample t test for difference of means](https://www.youtube.com/watch?v=Os-tcg9bF-c) — the §3.7 worked example.
- [Inference for categorical data (chi-square tests)](https://www.khanacademy.org/math/statistics-probability/inference-categorical-data-chi-square-tests) — chi-square unit, pairs with §3.6.
- [Chi-square statistic for hypothesis testing](https://www.youtube.com/watch?v=2QeDRsxSF9M) — the §3.6 statistic walked through with a numerical example.

> A good week: **two StatQuest videos for intuition (start with the p-value video and the Statistical Power video)**, **plus the Khan Academy chi-square unit with practice exercises**. That cements the spine of every later inference session in this course.

See you in **Session 04**, where we leave hypothesis testing behind and move into **regression** — predicting one number from another. The standard error and confidence interval ideas from this week will reappear, attached to regression coefficients.

<hr>

![](../_img/DK_Logo_White_150.png)

DataKolektiv, 2026.

[hello@datakolektiv.com](mailto:hello@datakolektiv.com)

<font size=1>License: [GPLv3](../LICENSE). This Notebook is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version. This Notebook is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.</font>